In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


nltk.download('stopwords')
nltk.download('punkt')

indonesian_stopwords = set(stopwords.words('indonesian'))

# Important words to KEEP for sentiment analysis
keep_words = {'tidak', 'jangan', 'bukan', 'tanpa', 'bagus', 'jelek', 'buruk', 'baik'}
custom_stopwords = indonesian_stopwords - keep_words

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sabrina\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sabrina\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# CLEANING
def clean_text(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove mentions (@)
    text = re.sub(r'@\w+', '', text)

    # Remove hashtag (#)
    text = re.sub(r'#', '', text)

    # Remove numbers (optional - keep if important)
    text = re.sub(r'\d+', '', text)
    
    # Remove emoji and non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    
    # Remove punctuation
    text = re.sub(r'[^\w\s\']', '', text)
    
    # Clean whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def preprocessing_traditional(text):
    # Cleaning
    text = clean_text(text)

    # Case folding (lowercase)
    text = text.lower()

    # Tokenization
    tokens = text.split()

    # Stopword removal BUT preserve short important words
    tokens = [word for word in tokens 
              if word not in custom_stopwords]
    
    return " ".join(tokens)

In [ ]:
# Load the data
df_all= pd.read_csv("../Normalize/02_combined_data.csv")

text_column = "content"

print(f"Data loaded successfully. Shape: {df_all.shape}")
print(f"Columns: {df_all.columns.tolist()}")
print(f"First row preview:\n{df_all.head()}")

Data loaded successfully. Shape: (6028, 13)
Columns: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'source', 'keyword']
First row preview:
                               reviewId         userName  \
0  6a7dbd15-2507-4f2c-bd2c-d36bbc89c640  Pengguna Google   
1  c7f1469f-42f5-4dad-86eb-13aca28e5cf7  Pengguna Google   
2  3bde987d-6726-4ed8-8cc7-de39a3331f68  Pengguna Google   
3  91616e30-ada7-4242-8872-ab3e310881f2  Pengguna Google   
4  beffc9f7-fa71-4ab9-9fe8-17c66bab72b0  Pengguna Google   

                                           userImage  \
0  https://play-lh.googleusercontent.com/EGemoI2N...   
1  https://play-lh.googleusercontent.com/EGemoI2N...   
2  https://play-lh.googleusercontent.com/EGemoI2N...   
3  https://play-lh.googleusercontent.com/EGemoI2N...   
4  https://play-lh.googleusercontent.com/EGemoI2N...   

                                             content  score  

In [ ]:
# Apply preprocessing
df_all["clean_text"] = df_all[text_column].apply(preprocessing_traditional)

# Remove duplicate rows based on reviewId only
subset_col = "reviewId"

# Show duplicate reviewId rows before removing them
if subset_col in df_all.columns:
    duplicates = df_all[df_all.duplicated(subset=[subset_col], keep=False)].sort_values(subset_col)
    print(f"Total duplicate rows found by reviewId: {len(duplicates)}")
    print(duplicates[[subset_col, text_column, "clean_text"]].head(20))
else:
    print(f"Column '{subset_col}' not found. Duplicate check skipped.")

# Remove duplicate rows after preprocessing
rows_before = len(df_all)
df_all = df_all.drop_duplicates(subset=[subset_col], keep="first").reset_index(drop=True)
print(f"Duplicate rows removed: {rows_before - len(df_all)}")

# Save the preprocessed data in the shared output folder
df_all.to_excel("../Output/03_combined_preprocessing.xlsx", index=False)

Total duplicate rows found by reviewId: 0
Empty DataFrame
Columns: [reviewId, content, clean_text]
Index: []
Duplicate rows removed: 0


In [ ]:
# Load the normalized data
df_all= pd.read_csv("../Output/02_combined_normalized_data.csv")

text_column = "content"

print(f"Data loaded successfully. Shape: {df_all.shape}")
print(f"Columns: {df_all.columns.tolist()}")
print(f"First row preview:\n{df_all.head()}")

Data loaded successfully. Shape: (6028, 4)
Columns: ['Unnamed: 0', 'reviewId', 'content', 'normalized']
First row preview:
   Unnamed: 0                              reviewId  \
0           0  6a7dbd15-2507-4f2c-bd2c-d36bbc89c640   
1           1  c7f1469f-42f5-4dad-86eb-13aca28e5cf7   
2           2  3bde987d-6726-4ed8-8cc7-de39a3331f68   
3           3  91616e30-ada7-4242-8872-ab3e310881f2   
4           4  beffc9f7-fa71-4ab9-9fe8-17c66bab72b0   

                                             content  \
0  masa Bank BUMN terbesar rating nya 4.1..tadiny...   
1  kenapa tidak bisa masuk ke aplikasi livin kami...   
2  heran aku, kejadian bukan 1 kali jadi di ATM s...   
3  Apk nya bagus cmn kebykn nyangkut di e-toll ud...   
4  knp kalo mau transfer harus upgrade ke gopay p...   

                                          normalized  
0  mengapa peringkat bank bumn terbesar demikian ...  
1  mengapa tidak bisa masuk ke aplikasi livin kam...  
2  saya heran kejadian ini bukan pertama kal

In [ ]:
# Apply preprocessing
df_all["clean_text"] = df_all[text_column].apply(preprocessing_traditional)

# Remove duplicate rows based on reviewId only
subset_col = "reviewId"

# Show duplicate reviewId rows before removing them
if subset_col in df_all.columns:
    duplicates = df_all[df_all.duplicated(subset=[subset_col], keep=False)].sort_values(subset_col)
    print(f"Total duplicate rows found by reviewId: {len(duplicates)}")
    print(duplicates[[subset_col, text_column, "clean_text"]].head(20))
else:
    print(f"Column '{subset_col}' not found. Duplicate check skipped.")

# Remove duplicate rows after preprocessing
rows_before = len(df_all)
df_all = df_all.drop_duplicates(subset=[subset_col], keep="first").reset_index(drop=True)
print(f"Duplicate rows removed: {rows_before - len(df_all)}")

# Save normalized preprocessed data in the shared output folder
df_all.to_excel("../Output/03_combined_normalized_preprocessing.xlsx", index=False)

Total duplicate rows found by reviewId: 0
Empty DataFrame
Columns: [reviewId, content, clean_text]
Index: []
Duplicate rows removed: 0
